# 7.3 Linear Layers, Activations, and Network Building Blocks

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook breaks down the standard components used to assemble feedforward neural networks. The goal is to make each layer feel concrete rather than magical.


## 7.3.1 Affine layers

### Why It Matters
A linear layer computes an affine transformation: matrix multiplication plus bias. The easiest way to understand it is to inspect its weight shapes and outputs.


In [ ]:
import torch
from torch import nn

layer = nn.Linear(3, 2)
x = torch.tensor([[1.0, 0.5, -1.0], [0.0, 2.0, 1.0]])
out = layer(x)

{
    "weight_shape": list(layer.weight.shape),
    "bias_shape": list(layer.bias.shape),
    "output_shape": list(out.shape),
}


### Interpretation
Every dense network is built from repeated affine transforms separated by nonlinearities.


## 7.3.2 ReLU, sigmoid, and tanh

### Why It Matters
Activation functions decide how the network bends and thresholds information. Looking at the same inputs through different activations makes their behavior easier to compare.


In [ ]:
import torch

x = torch.linspace(-2, 2, 5)
{
    "x": x.tolist(),
    "relu": torch.relu(x).tolist(),
    "sigmoid": torch.sigmoid(x).round(decimals=3).tolist(),
    "tanh": torch.tanh(x).round(decimals=3).tolist(),
}


### Interpretation
ReLU keeps positive values and clips negatives; sigmoid squashes to probabilities; tanh centers outputs around zero.


## 7.3.3 Stacking layers into MLPs

### Why It Matters
A multilayer perceptron is just a sequence of linear layers and activations. The value comes from composition: each layer transforms the representation further.


In [ ]:
import torch
from torch import nn

mlp = nn.Sequential(
    nn.Linear(4, 10),
    nn.ReLU(),
    nn.Linear(10, 6),
    nn.ReLU(),
    nn.Linear(6, 2),
)
x = torch.randn(3, 4)
out = mlp(x)
{
    "input_shape": list(x.shape),
    "output_shape": list(out.shape),
    "parameter_count": sum(p.numel() for p in mlp.parameters()),
}


### Interpretation
The stacked architecture is simple to read, which is why `nn.Sequential` is a useful teaching tool for feedforward models.


## 7.3.4 Hidden representations

### Why It Matters
Hidden layers create internal representations that are often more useful than raw inputs. Here we inspect the hidden activations for two points that should separate differently.


In [ ]:
import torch
from torch import nn

torch.manual_seed(13)
feature_extractor = nn.Sequential(nn.Linear(2, 3), nn.Tanh())
points = torch.tensor([[0.2, 0.1], [1.2, -0.8]])
hidden = feature_extractor(points)
hidden.round(decimals=3)


### Interpretation
The hidden vectors are the learned intermediate description of the inputs. Later layers work on that representation instead of the raw coordinates.


## 7.3.5 Output-layer design by task type

### Why It Matters
The final layer depends on the task. Regression often uses a raw scalar, binary classification uses one logit or probability, and multiclass classification uses one score per class.


In [ ]:
import torch
from torch import nn

x = torch.randn(5, 4)
regression_head = nn.Linear(4, 1)
binary_head = nn.Sequential(nn.Linear(4, 1), nn.Sigmoid())
multiclass_head = nn.Linear(4, 3)

{
    "regression_shape": list(regression_head(x).shape),
    "binary_range_example": binary_head(x)[:2].flatten().round(decimals=3).tolist(),
    "multiclass_shape": list(multiclass_head(x).shape),
}


### Interpretation
The head should reflect the output you need the model to produce, not just what is easy to code.


## 7.3.6 Build a small feedforward classifier

### Why It Matters
This subsection puts the pieces together into a small classifier and trains it on a synthetic binary problem.


In [ ]:
import torch
from torch import nn
from sklearn.datasets import make_moons

X_np, y_np = make_moons(n_samples=220, noise=0.15, random_state=14)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np.reshape(-1, 1), dtype=torch.float32)

model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.BCELoss()
for _ in range(300):
    opt.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward()
    opt.step()
with torch.no_grad():
    acc = (((model(X) > 0.5).float() == y).float().mean()).item()
{"training_accuracy": round(float(acc), 3), "final_loss": round(float(loss.item()), 4)}


### Interpretation
The architecture is still tiny, but it already demonstrates the standard classifier recipe used throughout neural-network practice.
